# Apex Retail Intelligence

## Notebook 01 : Raw & Landing Layer

### Objective

This notebook implements **Phase 1** and **Phase 2** of the Apex Retail Intelligence pipeline.

### Phase 1 – Raw Layer
- Read Historical and Incremental CSV files from Unity Catalog Volume
- Store files in the Raw zone
- Preserve all columns as StringType
- Keep Historical and Incremental datasets separate

### Phase 2 – Landing Layer
- Convert Raw CSV files to Parquet
- Validate record counts using audit files
- Generate PASS / FAIL audit report
- Stop execution if audit validation fails

### Technologies
- PySpark
- Databricks
- Unity Catalog Volume
- Parquet

In [0]:
# ============================================================
# Imports
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

import os
from datetime import datetime

In [0]:
# ============================================================
# Create Spark Session
# ============================================================

spark = (
    SparkSession.builder
    .appName("Apex Retail Intelligence - Raw & Landing")
    .getOrCreate()
)

print(f"Spark Version : {spark.version}")

Spark Version : 4.1.0


In [0]:
# ============================================================
# Project Configuration
# ============================================================

# Base Volume Path
BASE_PATH = "/Volumes/workspace/default/apex_retail_data"

# ------------------------------------------------------------
# Input Data Paths
# ------------------------------------------------------------

# Historical Data
HISTORICAL_CUSTOMER_PATH = f"{BASE_PATH}/historical/customer/customer_historical.csv"
HISTORICAL_PRODUCT_PATH  = f"{BASE_PATH}/historical/product/product_historical.csv"
HISTORICAL_SALES_PATH    = f"{BASE_PATH}/historical/sales/sales_historical.csv"

# Incremental Data
INCREMENTAL_CUSTOMER_PATH = f"{BASE_PATH}/incremental/customer/customer_incremental.csv"
INCREMENTAL_PRODUCT_PATH  = f"{BASE_PATH}/incremental/product/product_incremental.csv"
INCREMENTAL_SALES_PATH    = f"{BASE_PATH}/incremental/sales/sales_incremental.csv"

# ------------------------------------------------------------
# Landing Audit Files
# ------------------------------------------------------------

CUSTOMER_HIST_AUDIT_PATH = (
    f"{BASE_PATH}/audit_landing/customer/customer_historical_audit.csv"
)

CUSTOMER_INC_AUDIT_PATH = (
    f"{BASE_PATH}/audit_landing/customer/customer_incrementalaudit.csv"
)

PRODUCT_HIST_AUDIT_PATH = (
    f"{BASE_PATH}/audit_landing/product/product_historical_audit.csv"
)

PRODUCT_INC_AUDIT_PATH = (
    f"{BASE_PATH}/audit_landing/product/product_incrementalaudit.csv"
)

SALES_HIST_AUDIT_PATH = (
    f"{BASE_PATH}/audit_landing/sales/sales_historical_audit.csv"
)

SALES_INC_AUDIT_PATH = (
    f"{BASE_PATH}/audit_landing/sales/sales_incrementalaudit.csv"
)

# ------------------------------------------------------------
# Output Paths
# ------------------------------------------------------------

RAW_PATH = f"{BASE_PATH}/raw"

LANDING_PATH = f"{BASE_PATH}/landing"

In [0]:
# ============================================================
# Helper Functions
# ============================================================

def read_csv_as_string(file_path: str):
    """
    Reads a CSV file with header and infers all columns as StringType.

    Parameters:
        file_path (str): Path to the CSV file.

    Returns:
        DataFrame: Spark DataFrame with all columns as strings.
    """

    print(f"Reading file: {file_path}")

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")   # Keep all columns as StringType
        .csv(file_path)
    )

    return df


def read_audit_file(audit_path: str):
    """
    Reads an audit CSV file and returns the expected row count.

    Parameters:
        audit_path (str): Audit CSV path.

    Returns:
        int: Expected row count.
    """

    audit_df = (
        spark.read
        .option("header", "true")
        .csv(audit_path)
    )

    expected_count = int(audit_df.first()["row_count"])

    return expected_count


def validate_row_count(df, audit_path: str):
    """
    Validates DataFrame row count against the audit file.

    Raises:
        Exception if validation fails.
    """

    expected_count = read_audit_file(audit_path)
    actual_count = df.count()

    print(f"Expected Rows : {expected_count}")
    print(f"Actual Rows   : {actual_count}")

    if expected_count != actual_count:
        raise Exception(
            f"Audit Validation FAILED "
            f"(Expected={expected_count}, Actual={actual_count})"
        )

    print("Audit Validation PASSED")


def display_basic_info(df, dataset_name: str):
    """
    Displays basic dataset information.
    """

    print("=" * 60)
    print(f"Dataset : {dataset_name}")
    print("=" * 60)

    print(f"Rows    : {df.count()}")
    print(f"Columns : {len(df.columns)}")

    df.printSchema()

    display(df.limit(5))

In [0]:
# ============================================================
# Raw Layer Functions
# ============================================================

def write_raw_layer(df, output_path: str):
    """
    Writes the DataFrame to the Raw layer in CSV format.

    The Raw layer stores an exact copy of the source data
    without any transformations.
    """

    (
        df.write
        .mode("overwrite")
        .option("header", "true")
        .csv(output_path)
    )

    print(f"Raw data successfully written to:\n{output_path}")

In [0]:
# ============================================================
# Write Historical Customer to Raw Layer
# ============================================================

CUSTOMER_RAW_PATH = f"{RAW_PATH}/customer/historical"

write_raw_layer(
    customer_hist_df,
    CUSTOMER_RAW_PATH
)

Raw data successfully written to:
/Volumes/workspace/default/apex_retail_data/raw/customer/historical


In [0]:
# ============================================================
# Landing Layer Functions
# ============================================================

def write_landing_layer(df, output_path: str):
    """
    Writes the DataFrame to the Landing layer in Parquet format.

    Landing Layer Purpose:
    ----------------------
    - Stores data in columnar (Parquet) format.
    - Improves query performance.
    - Reduces storage size.
    - Preserves the original data (no transformations).
    """

    (
        df.write
        .mode("overwrite")
        .parquet(output_path)
    )

    print(f"Landing data successfully written to:\n{output_path}")

In [0]:
# ============================================================
# Write Historical Customer to Landing
# ============================================================

CUSTOMER_LANDING_PATH = f"{LANDING_PATH}/customer/historical"

write_landing_layer(
    customer_hist_df,
    CUSTOMER_LANDING_PATH
)

Landing data successfully written to:
/Volumes/workspace/default/apex_retail_data/landing/customer/historical


In [0]:
# ============================================================
# Verify Landing Data
# ============================================================

landing_customer_df = spark.read.parquet(
    CUSTOMER_LANDING_PATH
)

display_basic_info(
    landing_customer_df,
    "Landing Customer"
)

Dataset : Landing Customer
Rows    : 1052
Columns : 14
root
 |-- customer_id: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- income_bracket: string (nullable = true)
 |-- loyalty_program: string (nullable = true)
 |-- membership_years: string (nullable = true)
 |-- churned: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- number_of_children: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- customer_zip_code: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,occupation,customer_zip_code,customer_city,customer_state
1,56,Other,High,No,0,No,Divorced,3,Bachelor's,Self-Employed,37848,City D,State Y
2,69,Female,Medium,No,2,No,Married,2,PhD,Unemployed,44896,null,State X
3,46,Female,Low,No,5,No,Married,3,Bachelor's,Self-Employed,11816,City B,State X
4,32,Female,Low,No,0,No,Divorced,2,Master's,Employed,78604,City A,State Y
5,60,Female,null,Yes,7,Yes,Divorced,2,Bachelor's,Employed,17760,City B,State Z


In [0]:
# ============================================================
# End-to-End Dataset Processing
# ============================================================

def process_dataset(
    dataset_name: str,
    input_path: str,
    audit_path: str,
    raw_output_path: str,
    landing_output_path: str
):
    """
    End-to-end processing for a single dataset.

    Steps:
    1. Read CSV
    2. Display dataset information
    3. Validate audit
    4. Write Raw layer
    5. Write Landing layer
    """

    print("\n" + "=" * 70)
    print(f"Processing Dataset : {dataset_name}")
    print("=" * 70)

    # Read source CSV
    df = read_csv_as_string(input_path)

    # Display dataset information
    display_basic_info(df, dataset_name)

    # Validate audit file
    validate_row_count(df, audit_path)

    # Write Raw layer
    write_raw_layer(df, raw_output_path)

    # Write Landing layer
    write_landing_layer(df, landing_output_path)

    print(f"{dataset_name} processed successfully.\n")

    return df

In [0]:
# ============================================================
# Process Historical Datasets
# ============================================================

process_dataset(
    dataset_name="Customer Historical",
    input_path=HISTORICAL_CUSTOMER_PATH,
    audit_path=CUSTOMER_HIST_AUDIT_PATH,
    raw_output_path=f"{RAW_PATH}/customer/historical",
    landing_output_path=f"{LANDING_PATH}/customer/historical"
)

process_dataset(
    dataset_name="Product Historical",
    input_path=HISTORICAL_PRODUCT_PATH,
    audit_path=PRODUCT_HIST_AUDIT_PATH,
    raw_output_path=f"{RAW_PATH}/product/historical",
    landing_output_path=f"{LANDING_PATH}/product/historical"
)

process_dataset(
    dataset_name="Sales Historical",
    input_path=HISTORICAL_SALES_PATH,
    audit_path=SALES_HIST_AUDIT_PATH,
    raw_output_path=f"{RAW_PATH}/sales/historical",
    landing_output_path=f"{LANDING_PATH}/sales/historical"
)


Processing Dataset : Customer Historical
Reading file: /Volumes/workspace/default/apex_retail_data/historical/customer/customer_historical.csv
Dataset : Customer Historical
Rows    : 1052
Columns : 14
root
 |-- customer_id: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- income_bracket: string (nullable = true)
 |-- loyalty_program: string (nullable = true)
 |-- membership_years: string (nullable = true)
 |-- churned: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- number_of_children: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- customer_zip_code: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,occupation,customer_zip_code,customer_city,customer_state
1,56,Other,High,No,0,No,Divorced,3,Bachelor's,Self-Employed,37848,City D,State Y
2,69,Female,Medium,No,2,No,Married,2,PhD,Unemployed,44896,null,State X
3,46,Female,Low,No,5,No,Married,3,Bachelor's,Self-Employed,11816,City B,State X
4,32,Female,Low,No,0,No,Divorced,2,Master's,Employed,78604,City A,State Y
5,60,Female,null,Yes,7,Yes,Divorced,2,Bachelor's,Employed,17760,City B,State Z


Expected Rows : 1052
Actual Rows   : 1052
Audit Validation PASSED
Raw data successfully written to:
/Volumes/workspace/default/apex_retail_data/raw/customer/historical
Landing data successfully written to:
/Volumes/workspace/default/apex_retail_data/landing/customer/historical
Customer Historical processed successfully.


Processing Dataset : Product Historical
Reading file: /Volumes/workspace/default/apex_retail_data/historical/product/product_historical.csv
Dataset : Product Historical
Rows    : 1043
Columns : 16
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_brand: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_rating: string (nullable = true)
 |-- product_review_count: string (nullable = true)
 |-- product_stock: string (nullable = true)
 |-- product_return_rate: string (nullable = true)
 |-- product_size: string (nullable = true)
 |-- product_weight: string (nullable = true)
 |-- produc

product_id,product_name,product_brand,product_category,product_rating,product_review_count,product_stock,product_return_rate,product_size,product_weight,product_color,product_material,product_manufacture_date,product_expiry_date,product_shelf_life,unit_price
1480,Product D,Brand Y,Electronics,2.5,560,48,0.4,Small,4.61,Red,Metal,2019-08-04 01:47:01,2022-05-28 14:54:02,250,49.72
1597,Product C,Brand X,Groceries,4.7,413,80,0.3,Medium,0.84,Blue,Metal,2019-10-23 19:59:17,2022-12-19 08:04:41,180,817.76
5142,Product B,null,Toys,4.6,312,14,0.08,Medium,0.23,Green,Plastic,2018-05-12 08:00:29,2023-02-01 12:15:07,131,270.3
8447,Product A,Brand Z,Toys,1.1,110,69,0.09,Large,4.37,Blue,Wood,2019-11-15 16:17:29,2023-02-05 11:46:57,16,547.84
6025,Product C,Brand X,Clothing,3.8,172,25,0.39,Small,1.68,Red,Metal,2019-08-27 02:58:19,2023-10-05 08:13:07,57,785.29


Expected Rows : 1043
Actual Rows   : 1043
Audit Validation PASSED
Raw data successfully written to:
/Volumes/workspace/default/apex_retail_data/raw/product/historical
Landing data successfully written to:
/Volumes/workspace/default/apex_retail_data/landing/product/historical
Product Historical processed successfully.


Processing Dataset : Sales Historical
Reading file: /Volumes/workspace/default/apex_retail_data/historical/sales/sales_historical.csv
Dataset : Sales Historical
Rows    : 1002
Columns : 19
root
 |-- transaction_id: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_applied: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- store_location: string (nullable = true)
 |-- transaction_hour: string (nullable = true)
 |-- day_of_week: string (nullable =

transaction_id,transaction_date,customer_id,product_id,quantity,unit_price,discount_applied,payment_method,store_location,transaction_hour,day_of_week,week_of_year,month_of_year,total_sales,promotion_id,promotion_type,holiday_season,season,weekend
503290,2020-10-11 10:08:52,1,1480,8,49.72,0.5,Credit Card,Location A,18,Wednesday,27,7,563.16,271,20% Off,No,Spring,Yes
347796,2021-12-08 01:07:40,2,1597,7,817.76,0.32,Credit Card,Location C,15,Friday,20,2,7554.57,631,Flash Sale,No,Summer,Yes
493688,2020-02-17 09:40:48,3,5142,8,270.3,0.35,Debit Card,Location A,9,Saturday,35,6,7564.14,879,Flash Sale,Yes,Winter,Yes
861348,2020-08-13 00:43:14,4,8447,2,547.84,0.1,null,Location A,13,Friday,42,8,8125.92,211,Buy One Get One Free,Yes,Winter,No
535835,2021-07-02 11:59:03,5,6025,4,785.29,0.17,Mobile Payment,Location C,17,Monday,37,3,114.32,862,Flash Sale,Yes,Summer,Yes


Expected Rows : 1002
Actual Rows   : 1002
Audit Validation PASSED
Raw data successfully written to:
/Volumes/workspace/default/apex_retail_data/raw/sales/historical
Landing data successfully written to:
/Volumes/workspace/default/apex_retail_data/landing/sales/historical
Sales Historical processed successfully.



DataFrame[transaction_id: string, transaction_date: string, customer_id: string, product_id: string, quantity: string, unit_price: string, discount_applied: string, payment_method: string, store_location: string, transaction_hour: string, day_of_week: string, week_of_year: string, month_of_year: string, total_sales: string, promotion_id: string, promotion_type: string, holiday_season: string, season: string, weekend: string]

In [0]:
# ============================================================
# Process Incremental Datasets
# ============================================================

process_dataset(
    dataset_name="Customer Incremental",
    input_path=INCREMENTAL_CUSTOMER_PATH,
    audit_path=CUSTOMER_INC_AUDIT_PATH,
    raw_output_path=f"{RAW_PATH}/customer/incremental",
    landing_output_path=f"{LANDING_PATH}/customer/incremental"
)

process_dataset(
    dataset_name="Product Incremental",
    input_path=INCREMENTAL_PRODUCT_PATH,
    audit_path=PRODUCT_INC_AUDIT_PATH,
    raw_output_path=f"{RAW_PATH}/product/incremental",
    landing_output_path=f"{LANDING_PATH}/product/incremental"
)

process_dataset(
    dataset_name="Sales Incremental",
    input_path=INCREMENTAL_SALES_PATH,
    audit_path=SALES_INC_AUDIT_PATH,
    raw_output_path=f"{RAW_PATH}/sales/incremental",
    landing_output_path=f"{LANDING_PATH}/sales/incremental"
)


Processing Dataset : Customer Incremental
Reading file: /Volumes/workspace/default/apex_retail_data/incremental/customer/customer_incremental.csv
Dataset : Customer Incremental
Rows    : 1053
Columns : 19
root
 |-- customer_id: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- income_bracket: string (nullable = true)
 |-- loyalty_program: string (nullable = true)
 |-- membership_years: string (nullable = true)
 |-- churned: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- number_of_children: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- customer_zip_code: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- surrogate_key: string (nullable = true)
 |-- version: string (nullable = true)
 |-- effective_start_date: string (nullable = true)
 |-- effective_end_date: string 

customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,occupation,customer_zip_code,customer_city,customer_state,surrogate_key,version,effective_start_date,effective_end_date,is_current
1,56,Other,High,No,0,No,Divorced,3,Bachelor's,Self-Employed,37848,Old_City_1,Old_State_1,501,1,2020-01-01,2021-12-31,False
1,56,Other,High,No,0,No,Divorced,3,Bachelor's,Self-Employed,37848,New York,State NY,1,2,2022-01-01,null,True
2,69,Female,Medium,No,2,No,Married,2,PhD,Unemployed,44896,Old_City_2,Old_State_2,502,1,2020-01-01,2021-12-31,False
2,69,Female,Medium,No,2,No,Married,2,PhD,Unemployed,44896,Los Angeles,State CA,2,2,2022-01-01,null,True
3,46,Female,Low,No,5,No,Married,3,Bachelor's,Self-Employed,11816,City B,State X,3,1,2022-01-01,null,True


Expected Rows : 1053
Actual Rows   : 1053
Audit Validation PASSED
Raw data successfully written to:
/Volumes/workspace/default/apex_retail_data/raw/customer/incremental
Landing data successfully written to:
/Volumes/workspace/default/apex_retail_data/landing/customer/incremental
Customer Incremental processed successfully.


Processing Dataset : Product Incremental
Reading file: /Volumes/workspace/default/apex_retail_data/incremental/product/product_incremental.csv
Dataset : Product Incremental
Rows    : 1041
Columns : 17
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_brand: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_rating: string (nullable = true)
 |-- product_review_count: string (nullable = true)
 |-- product_stock: string (nullable = true)
 |-- product_return_rate: string (nullable = true)
 |-- product_size: string (nullable = true)
 |-- product_weight: string (nullable = true)
 |--

product_id,product_name,product_brand,product_category,product_rating,product_review_count,product_stock,product_return_rate,product_size,product_weight,product_color,product_material,product_manufacture_date,product_expiry_date,product_shelf_life,unit_price,last_updated
1480,Product D,Brand Y,Electronics,2.8,560,98,0.4,Small,4.61,Red,Metal,2019-08-04 01:47:01,2022-05-28 14:54:02,250,54.69,2026-04-17
1597,Product C,Brand X,Groceries,4.7,413,80,0.3,Medium,0.84,Blue,Metal,2019-10-23 19:59:17,2022-12-19 08:04:41,180,817.76,2026-04-17
5142,Product B,Brand X,Toys,4.6,312,14,0.08,Medium,0.23,Green,Plastic,2018-05-12 08:00:29,2023-02-01 12:15:07,131,270.3,2026-04-17
8447,Product A,Brand Z,Toys,1.4,110,119,0.09,Large,4.37,Blue,Wood,2019-11-15 16:17:29,2023-02-05 11:46:57,16,602.62,2026-04-17
6025,Product C,Brand X,Clothing,3.8,172,25,0.39,Small,1.68,Red,Metal,2019-08-27 02:58:19,2023-10-05 08:13:07,57,785.29,2026-04-17


Expected Rows : 1041
Actual Rows   : 1041
Audit Validation PASSED
Raw data successfully written to:
/Volumes/workspace/default/apex_retail_data/raw/product/incremental
Landing data successfully written to:
/Volumes/workspace/default/apex_retail_data/landing/product/incremental
Product Incremental processed successfully.


Processing Dataset : Sales Incremental
Reading file: /Volumes/workspace/default/apex_retail_data/incremental/sales/sales_incremental.csv
Dataset : Sales Incremental
Rows    : 1000
Columns : 19
root
 |-- transaction_id: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_applied: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- store_location: string (nullable = true)
 |-- transaction_hour: string (nullable = true)
 |-- day_of_week: string (nul

transaction_id,transaction_date,customer_id,product_id,quantity,unit_price,discount_applied,payment_method,store_location,transaction_hour,day_of_week,week_of_year,month_of_year,total_sales,promotion_id,promotion_type,holiday_season,season,weekend
5002714,2023-02-04 19:23:44,334,258,8,987.33,0.35,Credit Card,null,19,Monday,5,2,null,100,20% Off,No,Fall,No
5256471,2023-02-15 02:02:05,73,5708,2,519.75,0.41,Cash,Location D,2,Monday,7,2,613.31,358,null,null,Summer,null
6989689,2023-09-05 07:34:41,195,8719,2,806.52,0.47,Mobile Payment,Location A,7,null,36,9,854.91,475,20% Off,null,null,Yes
3145707,2023-11-20 04:12:02,831,5830,7,193.05,0.35,Credit Card,null,4,Thursday,47,11,878.38,967,Flash Sale,Yes,null,null
7253487,2023-01-26 14:42:18,379,5489,8,null,0.38,Mobile Payment,Location D,14,null,4,1,2821.02,476,Buy One Get One Free,No,Spring,null


Expected Rows : 1000
Actual Rows   : 1000
Audit Validation PASSED
Raw data successfully written to:
/Volumes/workspace/default/apex_retail_data/raw/sales/incremental
Landing data successfully written to:
/Volumes/workspace/default/apex_retail_data/landing/sales/incremental
Sales Incremental processed successfully.



DataFrame[transaction_id: string, transaction_date: string, customer_id: string, product_id: string, quantity: string, unit_price: string, discount_applied: string, payment_method: string, store_location: string, transaction_hour: string, day_of_week: string, week_of_year: string, month_of_year: string, total_sales: string, promotion_id: string, promotion_type: string, holiday_season: string, season: string, weekend: string]